# HBES Nuclear Decay Workbook
### Hydrogen Blip Exponential Scale — decay examples in QUIP / CHIP / BLIP units

This notebook demonstrates nuclear decay Q-values and half-lives expressed in the native units of the **HBES framework**, derived from the hydrogen 21 cm hyperfine transition.

| Unit | Definition | SI equivalent |
|------|-----------|---------------|
| **BLIP** | 10¹⁰ × τ(H) | ≈ 7.04 s |
| **QUIP** | h × f(H) | ≈ 9.41 × 10⁻²⁵ J |
| **CHIP** | QUIP / c² | ≈ 1.05 × 10⁻⁴¹ kg |

where τ(H) = 1/1,420,405,751.768 Hz is the period of the hydrogen hyperfine transition.

---
**Data source:** `protium_decays.json` — expandable lookup table.  
Add new entries to that file; this notebook recalculates HBES values automatically.

In [ ]:
# === HBES Constants ===
import sys, os

# Try to import from the hbes library (available when running inside the repo).
# Falls back to inline values if the package is not installed, so the notebook
# works as a standalone export without the source tree.
try:
    _repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if _repo_root not in sys.path:
        sys.path.insert(0, _repo_root)
    from hbes import H_FREQ, H_PLANCK, C, BLIP, QUIP, CHIP
    _source = 'hbes library'
except ImportError:
    # Inline fallback — exact values, SI 2019 / IAU 2015
    H_FREQ   = 1_420_405_751.768   # Hz   — hydrogen hyperfine frequency (exact by definition)
    H_PLANCK = 6.62607015e-34      # J·s  — Planck constant (exact, SI 2019)
    C        = 299_792_458.0       # m/s  — speed of light (exact, SI 2019)
    BLIP     = 1e10 / H_FREQ       # s    — 10¹⁰ × τ(H)
    QUIP     = H_PLANCK * H_FREQ   # J    — h × f(H), energy of one 21 cm photon
    CHIP     = QUIP / C**2         # kg   — QUIP/c²
    _source = 'inline fallback (hbes not found)'

# Convenience aliases and derived units
f_H     = H_FREQ
h       = H_PLANCK
c       = C
eV      = 1.602176634e-19     # J   — electron volt (exact, SI 2019)
MeV     = eV * 1e6

BLIP_s  = BLIP
QUIP_J  = QUIP
CHIP_kg = CHIP

print(f'HBES Base Units [{_source}]')
print(f'  1 BLIP = {BLIP_s:.6f} s')
print(f'  1 QUIP = {QUIP_J:.6e} J  ({QUIP_J/eV:.6e} eV)')
print(f'  1 CHIP = {CHIP_kg:.6e} kg')
print()
print('Self-referential check:')
print(f'  21 cm photon = {(h*f_H)/QUIP_J:.0f} QUIP  (exact by definition)')

In [ ]:
import json, math, os, sys

# Locate protium_decays.json. Three paths in priority order:
#   1. CWD/protium_decays.json        — Jupyter launched from notebook/
#   2. CWD/notebook/protium_decays.json — Jupyter launched from repo root
#   3. HTTP fetch from raw GitHub   — Google Colab (no local filesystem)
_GITHUB_RAW = (
    'https://raw.githubusercontent.com/patrixmyth/protium/main/protium_decays.json'
)

_candidates = [
    os.path.join(os.getcwd(), 'protium_decays.json'),
    os.path.join(os.getcwd(), 'notebook', 'protium_decays.json'),
]
_data_path = next((p for p in _candidates if os.path.isfile(p)), None)

if _data_path is not None:
    with open(_data_path) as _f:
        DECAY_DATA = json.load(_f)
    _source_label = os.path.basename(_data_path)
else:
    # Colab / offline export: fetch from GitHub
    try:
        import urllib.request
        with urllib.request.urlopen(_GITHUB_RAW) as _resp:
            DECAY_DATA = json.loads(_resp.read().decode())
        _source_label = 'GitHub (remote fetch)'
    except Exception as _e:
        raise FileNotFoundError(
            'protium_decays.json not found locally and remote fetch failed. '
            f'Error: {_e}\n'
            'Run Jupyter from notebook/ or the repo root, or check network access.'
        ) from _e

decays = DECAY_DATA['decays']

# Recompute HBES values from source-of-truth fields (Q_MeV, halflife_s).
# eV and MeV must be defined before this cell runs (see constants cell).
for d in decays:
    Q_J = d['Q_MeV'] * MeV
    d['Q_quips']            = Q_J / QUIP_J
    d['Q_quips_exp']        = math.log10(d['Q_quips'])
    d['delta_m_chips']      = d['Q_quips']       # numerically equal: CHIP ≡ QUIP/c²
    d['delta_m_chips_exp']  = d['Q_quips_exp']
    hl = d['halflife_s']
    d['halflife_blips']     = hl / BLIP_s
    d['halflife_blips_exp'] = math.log10(hl / BLIP_s) if hl > 0 else None

print(f'Loaded {len(decays)} decay entries from {_source_label}')
for d in decays:
    print(f"  {d['parent']} -> {d['daughter']}  [{d['mode']}]  Q={d['Q_MeV']} MeV  t1/2={d['halflife_human']}")


---
## Worked Example: Tritium Decay

Tritium (H-3) is the most self-referential decay in the HBES framework: hydrogen defines the unit system, and here hydrogen *decays*.

$$\mathrm{^3H} \rightarrow \mathrm{^3He} + e^- + \bar{\nu}_e \qquad Q = 18.591\,\mathrm{keV}$$

We express Q in **QUIPs**, mass deficit in **CHIPs**, and half-life in **BLIPs**.

In [ ]:
def worked_example(decay_id):
    """Full HBES worked example for a decay entry by id."""
    d = next(x for x in decays if x['id'] == decay_id)
    sep = '=' * 60
    print(sep)
    print(f"  {d['name']}")
    print(f"  {d['parent']} -> {d['daughter']}  [{d['mode']}]")
    print(sep)
    print()

    Q_J = d['Q_MeV'] * MeV
    print('Q-value (SI):')
    print(f"  Q = {d['Q_MeV']} MeV")
    print(f"    = {d['Q_MeV']} × 10⁶ × eV                  [1 MeV = 10⁶ eV]")
    print(f"    = {d['Q_MeV']} × 10⁶ × {eV:.6e} J/eV    [eV exact, SI 2019]")
    print(f"    = {Q_J:.4e} J")
    print()

    print('Q-value in QUIPs  [1 QUIP = h × f(H) = energy of one 21 cm photon]:')
    print(f"  Q = {Q_J:.4e} J ÷ {QUIP_J:.4e} J/QUIP")
    print(f"    = {d['Q_quips']:.4e} QUIPs")
    print(f"  log₁₀(Q/QUIP) = {d['Q_quips_exp']:.4f}")
    nearest = round(d['Q_quips_exp'])
    delta   = d['Q_quips_exp'] - nearest
    print(f"  ≈ 10^{nearest}  (Δ = {delta:+.4f} decades)")
    print()

    print('Mass deficit in CHIPs  [1 CHIP = QUIP / c² = h·f(H) / c²]:')
    print(f"  Δm = Q / c²")
    print(f"     = {Q_J:.4e} J ÷ ({c:.6e} m/s)²")
    print(f"     = {Q_J/c**2:.4e} kg")
    print(f"     = {d['delta_m_chips']:.4e} CHIPs")
    print(f"  log₁₀(Δm/CHIP) = {d['delta_m_chips_exp']:.4f}")
    print(f"  → same exponent as QUIPs: CHIP ≡ QUIP/c² absorbs the c² factor,")
    print(f"    so Δm in CHIPs is numerically identical to Q in QUIPs.")
    print()

    print('Half-life in BLIPs  [1 BLIP = 10¹⁰ × τ(H) ≈ 7.04 s]:')
    print(f"  t½ = {d['halflife_human']} = {d['halflife_s']:.4e} s")
    print(f"     = {d['halflife_s']:.4e} s ÷ {BLIP_s:.6f} s/BLIP")
    print(f"     = {d['halflife_blips']:.4e} BLIPs")
    if d['halflife_blips_exp'] is not None:
        nt = round(d['halflife_blips_exp'])
        dt = d['halflife_blips_exp'] - nt
        print(f"  log₁₀(t½/BLIP) = {d['halflife_blips_exp']:.4f}")
        print(f"  ≈ 10^{nt}  (Δ = {dt:+.4f} decades)")

    if d.get('notes'):
        print()
        print(f"Notes: {d['notes']}")
    print()

worked_example('H3-He3')

---
## Decay Explorer

Browse all table entries. Adjust `filter_mode` and `sort_by` to explore.

> **Note on `delta_m_chips` and `Q_quips`:** These columns always show the same number. This is not a data error — it is the framework's c = 1 property at the mass axis. In HBES units, 1 CHIP ≡ QUIP/c², so expressing the same energy as a mass deficit gives the same numerical value. The conversion E = Δm·c² becomes dimensionless: the c² factor is absorbed into the unit definition.

In [ ]:
filter_mode = None           # None = all, or 'alpha' 'beta-' 'beta+' 'gamma'
sort_by     = 'Q_quips_exp'  # 'Q_quips_exp' | 'halflife_blips_exp' | 'name'

subset = [d for d in decays if filter_mode is None or d['mode'] == filter_mode]
subset = sorted(subset, key=lambda x: x.get(sort_by) or 0)

hdr = f"{'Decay':<30} {'Mode':<8} {'Q (MeV)':<12} {'log10(Q/QUIP)':<16} {'t1/2':<14} {'log10(t1/2/BLIP)'}"
print(hdr)
print('-' * len(hdr))
for d in subset:
    texp = f"{d['halflife_blips_exp']:.3f}" if d['halflife_blips_exp'] else '---'
    print(f"{d['parent']+' -> '+d['daughter']:<30}{d['mode']:<8}{d['Q_MeV']:<12}{d['Q_quips_exp']:<16.4f}{d['halflife_human']:<14}{texp}")

In [ ]:
# Change the id to any entry in the table above
worked_example('Bi212-Tl208')

---
## HBES Log-Scale Position

Each decay plotted on the QUIP (energy) and BLIP (time) log axes.

In [ ]:
import matplotlib.pyplot as plt

colors = {'alpha': '#e05c2a', 'beta-': '#2a7ae0', 'beta+': '#2ae07a', 'gamma': '#9b2ae0'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Nuclear Decays on the HBES Log Scale', fontsize=14, fontweight='bold')

for ax, key, label in [
    (axes[0], 'Q_quips_exp',        'log10(Q / QUIP)'),
    (axes[1], 'halflife_blips_exp',  'log10(t1/2 / BLIP)'),
]:
    for d in decays:
        val = d.get(key)
        if val is None: continue
        col = colors.get(d['mode'], 'grey')
        ax.scatter(val, 0, color=col, s=120, zorder=3)
        ax.annotate(d['parent'], (val, 0),
                    textcoords='offset points', xytext=(0, 12),
                    ha='center', fontsize=8, rotation=45)
    ax.set_xlabel(label, fontsize=11)
    ax.set_yticks([])
    ax.grid(axis='x', alpha=0.3)

axes[0].axvline(0, color='gold', linestyle='--', linewidth=1.5, label='1 QUIP = 1 × 21cm photon')
axes[0].set_title('Q-value (QUIPs)')
axes[0].legend(fontsize=8)
axes[1].set_title('Half-life (BLIPs)')

from matplotlib.lines import Line2D
leg = [Line2D([0],[0], marker='o', color='w', markerfacecolor=v, markersize=9, label=k)
       for k, v in colors.items()]
fig.legend(handles=leg, loc='lower center', ncol=4, fontsize=9, title='Decay mode')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

---
## Adding New Entries

Add a JSON object to the `decays` array in `protium_decays.json`:

```json
{
  "id":             "unique-string",
  "name":           "Human-readable name",
  "parent":         "Po-210",
  "daughter":       "Pb-206",
  "mode":           "alpha",
  "Q_MeV":          5.3044,
  "halflife_s":     11955168,
  "halflife_human": "138.4 days",
  "notes":          "optional context"
}
```

The notebook recomputes all HBES quantities from `Q_MeV` and `halflife_s` — no manual calculation needed.

**Data sources:**
- [NUBASE2020](https://www.sciencedirect.com/science/article/pii/S1674198120001458)
- [NNDC Chart of Nuclides](https://www.nndc.bnl.gov/nudat3/)
- [IAEA Live Chart](https://www-nds.iaea.org/relnsd/vcharthtml/VChartHTML.html)